In [ ]:
%cd C:/Users/dirk1/0.SYNTRA1/syntra_ds1/whosethat1

In [1]:

whosethat1/
  app.py
  requirements.txt
  README.md
  .gitignore
  data/
    silhouettes/        (auto-generated)
    meta.csv            (auto-generated)
    features.npy        (auto-generated)

UsageError: the following arguments are required: filename


In [2]:
#create tree strucrure

from pathlib import Path

root = Path("whosethat1")
(root / "data" / "silhouettes").mkdir(parents=True, exist_ok=True)

for fn in ["app.py", "requirements.txt", "README.md", ".gitignore"]:
    (root / fn).touch(exist_ok=True)

print("Created:", root.resolve())

Created: C:\Users\dirk1\0.SYNTRA1\syntra_ds1\whosethat1\whosethat1


In [3]:
import streamlit as st
import pandas as pd
import random
from PIL import Image, ImageDraw, ImageFilter
import io

# Set page config
st.set_page_config(
    page_title="Who's That Pokémon?",
    page_icon="🎮",
    layout="wide"
)

# Title
st.title("Who's That Pokémon? 🔍")
st.markdown("### Can you guess the Pokémon from its silhouette?")

# Pokémon database (simplified for the assignment)
# In a real app, you might want to use a more comprehensive dataset
POKEMON = {
    "Pikachu": {
        "color": "#FFD700",
        "difficulty": "easy",
        "hint": "Electric mouse"
    },
    "Bulbasaur": {
        "color": "#7CFC00",
        "difficulty": "easy",
        "hint": "Has a seed on its back"
    },
    "Charmander": {
        "color": "#FF4500",
        "difficulty": "easy",
        "hint": "Fire lizard with a flame on its tail"
    },
    "Squirtle": {
        "color": "#1E90FF",
        "difficulty": "easy",
        "hint": "Tiny turtle Pokémon"
    },
    "Jigglypuff": {
        "color": "#FFB6C1",
        "difficulty": "medium",
        "hint": "Balloon-shaped and sings"
    },
    "Meowth": {
        "color": "#F0E68C",
        "difficulty": "medium",
        "hint": "Coin on its forehead"
    },
    "Psyduck": {
        "color": "#FFFF00",
        "difficulty": "medium",
        "hint": "Always has a headache"
    },
    "Snorlax": {
        "color": "#00008B",
        "difficulty": "hard",
        "hint": "Sleeping giant"
    },
    "Mewtwo": {
        "color": "#9370DB",
        "difficulty": "hard",
        "hint": "Genetically created legendary"
    },
    "Gengar": {
        "color": "#4B0082",
        "difficulty": "hard",
        "hint": "Shadow Pokémon"
    }
}

def create_silhouette(name):
    """Create a simple silhouette image for a Pokémon"""
    # Create a blank image with a white background
    img = Image.new('RGB', (300, 300), color='white')
    draw = ImageDraw.Draw(img)
    
    # Draw a simple shape based on the Pokémon
    # This is a simplified representation - in a real app you'd use actual images
    if name == "Pikachu":
        # Draw a simple Pikachu-like shape (oval with ears)
        draw.ellipse([100, 80, 200, 180], fill='black')  # head
        draw.ellipse([120, 40, 140, 80], fill='black')  # left ear
        draw.ellipse([160, 40, 180, 80], fill='black')  # right ear
        draw.ellipse([130, 100, 150, 120], fill='white')  # left eye (white part)
        draw.ellipse([150, 100, 170, 120], fill='white')  # right eye (white part)
        draw.ellipse([140, 120, 160, 140], fill='black')  # nose
    elif name == "Bulbasaur":
        draw.ellipse([100, 100, 200, 200], fill='black')  # body
        draw.ellipse([140, 50, 160, 100], fill='black')  # head
        draw.ellipse([130, 40, 170, 60], fill='black')  # bulb
    elif name == "Charmander":
        draw.ellipse([120, 100, 180, 160], fill='black')  # body
        draw.ellipse([140, 60, 160, 100], fill='black')  # head
        draw.polygon([150, 40, 140, 60, 160, 60], fill='black')  # flame
    elif name == "Squirtle":
        draw.ellipse([110, 90, 190, 170], fill='black')  # body
        draw.ellipse([140, 50, 160, 90], fill='black')  # head
        draw.arc([120, 80, 180, 120], 0, 180, fill='black', width=5)  # shell
    else:
        # Generic shape for other Pokémon
        draw.ellipse([100, 80, 200, 180], fill='black')
        draw.ellipse([130, 40, 170, 80], fill='black')
    
    return img

def main():
    # Initialize session state
    if 'current_pokemon' not in st.session_state:
        st.session_state.current_pokemon = random.choice(list(POKEMON.keys()))
    if 'score' not in st.session_state:
        st.session_state.score = 0
    if 'attempts' not in st.session_state:
        st.session_state.attempts = 0
    if 'show_hint' not in st.session_state:
        st.session_state.show_hint = False
    if 'guessed' not in st.session_state:
        st.session_state.guessed = False

    # Create two columns
    col1, col2 = st.columns([1, 1])

    with col1:
        st.markdown("### 🔍 Pokémon Silhouette")
        
        # Get current Pokémon data
        current = st.session_state.current_pokemon
        pokemon_data = POKEMON[current]
        
        # Create and display silhouette
        silhouette = create_silhouette(current)
        st.image(silhouette, caption="Who's that Pokémon?", use_container_width=True)
        
        # Game stats
        st.markdown(f"**Score:** {st.session_state.score}")
        st.markdown(f"**Attempts:** {st.session_state.attempts}")
        st.markdown(f"**Difficulty:** {pokemon_data['difficulty'].title()}")

    with col2:
        st.markdown("### 🤔 Your Guess")
        
        # Guess input
        guess = st.text_input("Enter Pokémon name:", key="guess_input", 
                             disabled=st.session_state.guessed)
        
        # Buttons
        col_guess, col_hint, col_new = st.columns(3)
        
        with col_guess:
            if st.button("Submit Guess", disabled=st.session_state.guessed):
                if guess:
                    st.session_state.attempts += 1
                    if guess.strip().lower() == current.lower():
                        st.success(f"✅ Correct! It's {current}!")
                        st.session_state.score += 1
                        st.session_state.guessed = True
                        st.balloons()
                    else:
                        st.error(f"❌ Sorry, that's not {current}")
        
        with col_hint:
            if st.button("Show Hint"):
                st.session_state.show_hint = True
        
        with col_new:
            if st.button("New Pokémon"):
                st.session_state.current_pokemon = random.choice(list(POKEMON.keys()))
                st.session_state.show_hint = False
                st.session_state.guessed = False
                st.rerun()
        
        # Display hint
        if st.session_state.show_hint:
            st.info(f"💡 Hint: {pokemon_data['hint']}")
        
        # Show answer if guessed
        if st.session_state.guessed:
            st.success(f"It was {current}!")
            
            # Color reveal
            st.markdown(f"**Color:** {pokemon_data['color']}")
            
            # Option to play again
            if st.button("Play Again"):
                st.session_state.current_pokemon = random.choice(list(POKEMON.keys()))
                st.session_state.show_hint = False
                st.session_state.guessed = False
                st.rerun()

    # Game instructions
    with st.expander("How to Play"):
        st.markdown("""
        - A silhouette of a Pokémon will be displayed
        - Type your guess in the text box
        - Click 'Submit Guess' to check your answer
        - Need help? Click 'Show Hint' for a clue
        - Get a new Pokémon with 'New Pokémon'
        - Try to guess correctly to increase your score!
        """)

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'streamlit'